In [ ]:
%cd ..

In [ ]:
ENTITY = input("Your wandb entity")
SWEEP_ID = input("The wandb sweep ID of the induction experiment")

import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
import json
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial

def fetch_run_data(run):
    """Fetch data for a single run - to be run in parallel."""
    try:
        stats = dict(run.config)
        stats.update(run.summary)
        stats["run_id"] = run.id
        stats["state"] = run.state
        return stats
    except Exception as e:
        print(f"Error fetching run {run.id}: {e}")
        return None

def load_sweep(sweep_id, max_runs=1000000000, max_workers=10, batch_size=100):
    """
    Load sweep runs in parallel for faster processing.
    
    Args:
        sweep_id: W&B sweep ID
        max_runs: Maximum number of runs to fetch
        max_workers: Number of parallel workers (threads) for fetching run details
        batch_size: Number of runs to fetch before processing in parallel
    """
    api = wandb.Api(timeout=100)

    warnings.filterwarnings("ignore", message="DataFrame is highly fragmented")

    # Collect configs and full history for each run
    all_runs_data = []

    # Use api.runs() with filters to stream runs
    runs = api.runs(
        path=f"{ENTITY}/softmax",
        filters={"sweep": sweep_id},
        per_page=50
    )

    # Convert to list iterator so we can batch it
    runs_iter = iter(runs)
    total_processed = 0
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        while total_processed < max_runs:
            # Get a batch of runs
            batch = []
            for _ in range(batch_size):
                try:
                    run = next(runs_iter)
                    batch.append(run)
                    total_processed += 1
                    if total_processed >= max_runs:
                        break
                except StopIteration:
                    break
            
            if not batch:
                break
            
            # Process batch in parallel
            futures = {executor.submit(fetch_run_data, run): run for run in batch}
            
            # Collect results with progress bar
            for future in tqdm(as_completed(futures), total=len(futures), 
                             desc=f"Processing runs {total_processed-len(batch)+1}-{total_processed}"):
                result = future.result()
                if result is not None:
                    all_runs_data.append(result)

    # Concatenate all runs into one long DataFrame
    df = pd.DataFrame(all_runs_data)

    # Display info about the DataFrame
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    return df

In [ ]:
data = load_sweep(SWEEP_ID)
data.to_csv("data/induction_data.csv", index=False)

# MAIN

In [ ]:
import pandas as pd

data = pd.read_csv("data/induction_data.csv")
data = data[data["attention_type"] != "uniform"]
data["attention_type"] = data["attention_type"].apply(lambda x: "linear_norm" if x == "linear" else x)
data.head()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ICML-style settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

NORMALIZED = "w. norm."
UNNORMALIZED = "w/o norm."

# Filter for runs where eval/acc_induction > 0.95
high_acc_runs = data[data['eval/acc_induction'] > 0.95].copy()

# Find all columns matching pattern 1_<number>_rel_attn_bos_global
import re
pattern = re.compile(r'^1_\d+_rel_attn_bos_global$')
rel_attn_cols = [col for col in high_acc_runs.columns if pattern.match(col)]

# Set threshold (adjust as needed)
THRESHOLD = 0.9

# For each run, compute the proportion of heads with sink score > THRESHOLD
def compute_proportion_above_threshold(row):
    values = row[rel_attn_cols].dropna()
    if len(values) == 0:
        return np.nan
    return (values > THRESHOLD).sum() / len(values)

high_acc_runs['proportion_above_threshold'] = high_acc_runs.apply(compute_proportion_above_threshold, axis=1)

# Filter for specific conditions
subset = high_acc_runs[
    (high_acc_runs['aux_loss_weight'] == 0.2) & 
    (high_acc_runs['shift_query_token'] == True) &
    (high_acc_runs['positional_type'] == 'add')
].copy()

# Extract base attention type and norm/unnorm status
def extract_base_and_norm(attn_type):
    if attn_type == 'softmax':
        return 'softmax', NORMALIZED
    elif attn_type.endswith('_norm'):
        return attn_type[:-5], NORMALIZED
    elif attn_type.endswith('_unnorm'):
        return attn_type[:-7], UNNORMALIZED
    else:
        return attn_type, None

subset['base_attention_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[0])
subset['norm_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[1])

# Rename attention types
rename_map = {
    'linear': r'elu $\cdot$ elu',
    'coupled_linear': 'elu',
    'raw_dotproduct': 'linear',
    'softmax': 'softmax',
    'sigmoid': 'sigmoid',
    'mlp': 'mlp',
}
subset['base_attention_type'] = subset['base_attention_type'].map(lambda x: rename_map.get(x, x))

# Keep only attention types that have both norm and unnorm variants, plus softmax
base_types_with_both = subset.groupby('base_attention_type')['norm_type'].apply(
    lambda x: set(x.dropna()) == {NORMALIZED, UNNORMALIZED}
)
valid_base_types = base_types_with_both[base_types_with_both].index.tolist()
if 'softmax' not in valid_base_types:
    valid_base_types.append('softmax')
subset = subset[subset['base_attention_type'].isin(valid_base_types)]
subset = subset[subset['norm_type'].notna()]

# HLS palette colors 3 and 5
hls = sns.color_palette("hls", 8)
colors = [hls[3], hls[5]]

# ICML single column width, shorter height
fig, ax = plt.subplots(figsize=(3.25, 1.8))

# Order as specified
desired_order = ['softmax', 'sigmoid', 'elu', r'elu $\cdot$ elu', 'mlp', 'linear']
order = [t for t in desired_order if t in subset['base_attention_type'].unique()]

sns.barplot(
    data=subset,
    x='base_attention_type',
    y='proportion_above_threshold',
    hue='norm_type',
    hue_order=[NORMALIZED, UNNORMALIZED],
    order=order,
    palette=colors,
    ax=ax,
    errorbar=None,
    edgecolor='black',
    linewidth=0.5,
    width=0.7,
    gap=0.1,
)

# Shift softmax bar to center
softmax_idx = order.index('softmax') if 'softmax' in order else -1
if softmax_idx >= 0:
    for i, patch in enumerate(ax.patches):
        if i == softmax_idx:
            x = patch.get_x()
            width = patch.get_width()
            patch.set_x(x + width / 1.7)

ax.set_xlabel('')
ax.set_ylabel('% of sinks')
ax.set_ylim(0, 1)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
ax.tick_params(axis='x', rotation=45)

# Clean up legend
ax.legend(title='', frameon=False, loc='upper right')

plt.tight_layout()
# plt.savefig('sink_scores.pdf')
plt.show()

In [ ]:
import os
from itertools import product

# ICML-style settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

NORMALIZED = "w. norm."
UNNORMALIZED = "w/o norm."
THRESHOLD_SINK_SCORE = 0.9
THRESHOLD_ACC = 0.95

# Filter for runs where eval/acc_induction > THRESHOLD_ACC
high_acc_runs = data[data['eval/acc_induction'] > THRESHOLD_ACC].copy()

# Find all columns matching pattern 1_<number>_rel_attn_bos_global
pattern = re.compile(r'^1_\d+_rel_attn_bos_global$')
rel_attn_cols = [col for col in high_acc_runs.columns if pattern.match(col)]

def compute_proportion_above_threshold(row):
    values = row[rel_attn_cols].dropna()
    if len(values) == 0:
        return np.nan
    return (values > THRESHOLD_SINK_SCORE).sum() / len(values)

high_acc_runs['proportion_above_threshold'] = high_acc_runs.apply(compute_proportion_above_threshold, axis=1)

def extract_base_and_norm(attn_type):
    if attn_type == 'softmax':
        return 'softmax', NORMALIZED
    elif attn_type.endswith('_norm'):
        return attn_type[:-5], NORMALIZED
    elif attn_type.endswith('_unnorm'):
        return attn_type[:-7], UNNORMALIZED
    else:
        return attn_type, None

rename_map = {
    'linear': r'elu $\cdot$ elu',
    'coupled_linear': 'elu',
    'raw_dotproduct': 'linear',
    'softmax': 'softmax',
    'sigmoid': 'sigmoid',
    'mlp': 'mlp',
}

hls = sns.color_palette("hls", 8)
colors = [hls[3], hls[5]]
desired_order = ['softmax', 'sigmoid', 'elu', r'elu $\cdot$ elu', 'mlp', 'linear']

# Get unique values
aux_loss_weights = sorted(high_acc_runs['aux_loss_weight'].unique())
shift_query_tokens = sorted(high_acc_runs['shift_query_token'].unique())
positional_types = sorted(high_acc_runs['positional_type'].unique())

output_dir = 'visualizations/fig'

for aux_loss, shift_query, pos_type in product(aux_loss_weights, shift_query_tokens, positional_types):
    subset = high_acc_runs[
        (high_acc_runs['aux_loss_weight'] == aux_loss) & 
        (high_acc_runs['shift_query_token'] == shift_query) &
        (high_acc_runs['positional_type'] == pos_type)
    ].copy()
    
    subset['base_attention_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[0])
    subset['norm_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[1])
    subset['base_attention_type'] = subset['base_attention_type'].map(lambda x: rename_map.get(x, x))
    subset = subset[subset['norm_type'].notna()]
    
    fig, ax = plt.subplots(figsize=(3.25, 1.8))
    
    # Always use full order
    order = desired_order
    
    # Track which combinations have data
    existing_combinations = set()
    for _, row in subset.iterrows():
        existing_combinations.add((row['base_attention_type'], row['norm_type']))
    
    # Plot bars if there's data
    if len(subset) > 0:
        sns.barplot(
            data=subset,
            x='base_attention_type',
            y='proportion_above_threshold',
            hue='norm_type',
            hue_order=[NORMALIZED, UNNORMALIZED],
            order=order,
            palette=colors,
            ax=ax,
            errorbar=None,
            edgecolor='black',
            linewidth=0.5,
            width=0.7,
            gap=0.1,
        )
    
    # Calculate bar positions and add red crosses for missing data
    n_hues = 2
    width = 0.7
    gap = 0.1
    bar_width = width / n_hues - gap / n_hues
    
    for i, attn_type in enumerate(order):
        for j, norm_type in enumerate([NORMALIZED, UNNORMALIZED]):
            # Skip softmax unnormalized (doesn't exist by design)
            if attn_type == 'softmax' and norm_type == UNNORMALIZED:
                continue
            
            if (attn_type, norm_type) not in existing_combinations:
                # Calculate x position for the missing bar
                x_pos = i + (j - 0.5) * (bar_width + gap / 2)
                # Add red cross
                ax.plot(x_pos, 0.1, 'x', color='red', markersize=7, markeredgewidth=2)
    
    # Shift softmax bar to center if it exists
    softmax_idx = order.index('softmax')
    if ('softmax', NORMALIZED) in existing_combinations:
        for idx, patch in enumerate(ax.patches):
            if idx == softmax_idx:
                x = patch.get_x()
                w = patch.get_width()
                patch.set_x(x + w / 1.7)
    
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order)
    ax.set_xlabel('')
    ax.set_ylabel('% of sink heads')
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
    ax.tick_params(axis='x', rotation=45)
    
    # Only show legend if we have bars
    if len(subset) > 0:
        legend = ax.legend(title='', frameon=False, loc='upper right')

        # Render to get proper bounding boxes
        fig.canvas.draw()
        
        # Get legend bbox in data coordinates
        legend_bbox = legend.get_window_extent().transformed(ax.transData.inverted())
        
        # Check if any bar overlaps with legend
        overlaps = False
        for patch in ax.patches:
            bar_height = patch.get_height()
            bar_x = patch.get_x()
            bar_w = patch.get_width()
            # Check overlap: bar extends into legend region
            if (bar_x + bar_w > legend_bbox.x0 and 
                bar_x < legend_bbox.x1 and 
                bar_height > legend_bbox.y0):
                overlaps = True
                break
        
        if overlaps:
            legend.remove()
    
    plt.tight_layout()
    
    # Save with descriptive filename
    shift_str = "shift" if shift_query else "noshift"
    filename = f"sink_proportions_aux{aux_loss}_{shift_str}_{pos_type}.pdf"
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()
    print(f"Saved: {filename}")

print("Done!")

In [ ]:
import os
from itertools import product

# ICML-style settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times', 'Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'hatch.linewidth': 0.1,
})

NORMALIZED = "w. norm."
UNNORMALIZED = "w/o norm."

def extract_base_and_norm(attn_type):
    if attn_type == 'softmax':
        return 'softmax', NORMALIZED
    elif attn_type.endswith('_norm'):
        return attn_type[:-5], NORMALIZED
    elif attn_type.endswith('_unnorm'):
        return attn_type[:-7], UNNORMALIZED
    else:
        return attn_type, None

rename_map = {
    'linear': r'elu $\cdot$ elu',
    'coupled_linear': 'elu',
    'raw_dotproduct': 'linear',
    'softmax': 'softmax',
    'sigmoid': 'sigmoid',
    'mlp': 'mlp',
}

hls = sns.color_palette("hls", 8)
colors = [hls[3], hls[5]]
desired_order = ['softmax', 'sigmoid', 'elu', r'elu $\cdot$ elu', 'mlp', 'linear']

# Get unique values from full data
aux_loss_weights = sorted(data['aux_loss_weight'].unique())
shift_query_tokens = sorted(data['shift_query_token'].unique())
positional_types = sorted(data['positional_type'].unique())

output_dir = 'visualizations/fig'

for aux_loss, shift_query, pos_type in product(aux_loss_weights, shift_query_tokens, positional_types):
    subset = data[
        (data['aux_loss_weight'] == aux_loss) & 
        (data['shift_query_token'] == shift_query) &
        (data['positional_type'] == pos_type)
    ].copy()
    
    subset['base_attention_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[0])
    subset['norm_type'] = subset['attention_type'].apply(lambda x: extract_base_and_norm(x)[1])
    subset['base_attention_type'] = subset['base_attention_type'].map(lambda x: rename_map.get(x, x))
    subset = subset[subset['norm_type'].notna()]
    
    fig, ax = plt.subplots(figsize=(3.25, 1.8))
    
    # Always use full order
    order = desired_order
    
    # Track which combinations have data
    existing_combinations = set()
    for _, row in subset.iterrows():
        existing_combinations.add((row['base_attention_type'], row['norm_type']))
    
    # Plot bars if there's data
    if len(subset) > 0:
        sns.barplot(
            data=subset,
            x='base_attention_type',
            y='eval/acc_induction',
            hue='norm_type',
            hue_order=[NORMALIZED, UNNORMALIZED],
            order=order,
            palette=colors,
            ax=ax,
            errorbar=None,
            edgecolor='black',
            linewidth=0.5,
            width=0.7,
            gap=0.1,
        )
        
        # Add subtle horizontal line hatching to all bars
        for patch in ax.patches:
            patch.set_hatch('xxxxxxxxx')
    
    # Calculate bar positions and add red crosses for missing data
    n_hues = 2
    width = 0.7
    gap = 0.1
    bar_width = width / n_hues - gap / n_hues
    
    for i, attn_type in enumerate(order):
        for j, norm_type in enumerate([NORMALIZED, UNNORMALIZED]):
            # Skip softmax unnormalized (doesn't exist by design)
            if attn_type == 'softmax' and norm_type == UNNORMALIZED:
                continue
            
            if (attn_type, norm_type) not in existing_combinations:
                # Calculate x position for the missing bar
                x_pos = i + (j - 0.5) * (bar_width + gap / 2)
                # Add red cross
                ax.plot(x_pos, 0.1, 'x', color='red', markersize=7, markeredgewidth=2)
    
    # Shift softmax bar to center if it exists
    softmax_idx = order.index('softmax')
    if ('softmax', NORMALIZED) in existing_combinations:
        for idx, patch in enumerate(ax.patches):
            if idx == softmax_idx:
                x = patch.get_x()
                w = patch.get_width()
                patch.set_x(x + w / 1.7)
    
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order)
    ax.set_xlabel('')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
    ax.tick_params(axis='x', rotation=45)
    
    # Add legend and check for overlap with bars
    if len(subset) > 0:
        legend = ax.legend(title='', frameon=False, loc='upper left', ncol=2)
        
        # Render to get proper bounding boxes
        fig.canvas.draw()
        
        # Get legend bbox in data coordinates
        legend_bbox = legend.get_window_extent().transformed(ax.transData.inverted())
        
        # Check if any bar overlaps with legend
        overlaps = False
        for patch in ax.patches:
            bar_height = patch.get_height()
            bar_x = patch.get_x()
            bar_w = patch.get_width()
            # Check overlap: bar extends into legend region
            if (bar_x + bar_w > legend_bbox.x0 and 
                bar_x < legend_bbox.x1 and 
                bar_height > legend_bbox.y0):
                overlaps = True
                break
        
        if overlaps:
            legend.remove()
    
    plt.tight_layout()
    
    # Save with descriptive filename
    shift_str = "shift" if shift_query else "noshift"
    filename = f"accuracy_aux{aux_loss}_{shift_str}_{pos_type}.pdf"
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()
    print(f"Saved: {filename}")

print("Done!")